# P4 / 02: S05 residual at 15 minutes without entries in earnings windows (both windows)

Protocol: [`001_claude_p4_news_filtered_reversal_preregistration.md`](../../discussions/News_Filtered_Reversal/001_claude_p4_news_filtered_reversal_preregistration.md), section 6. S05 0.1.3, `x10` exits, measured costs. Arm `EX`: no new entries on sessions inside an earnings window {f−1, f, f+1} of that stock. `ALL`: no mask.

**Gates** per window (α = 0.05):
- **Q1** outside − inside gross per trade in `ALL` > 0;
- **Q2** `EX` beats 100 random exclusions of the same size per session (net return and Sharpe);
- **Q3** `EX` net > 0 at 1× and 2×, bootstrap lower bound > 0.

`EDGE_CANDIDATE` only if all pass in **both** windows.

In [1]:
N_PLACEBO = 100
N_JOBS = -1

import json, os, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, p1, p2, p4, s04, s05, s05b
from equity_trading.alpaca import ALPACA_VERSION, download_years
from equity_trading.data import vendor_dir, work_dir

SYNTHETIC = os.environ.get("P4_SYNTHETIC") == "1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
CFG = p4.P4Config()
WINDOWS = {"A 2020-2025": ("2020-01-01", "2025-12-31"), "B 2017-2019": ("2017-01-01", "2019-12-31")}   # 2026 sealed
QUOTE_CACHE = vendor_dir("Alpaca") / "quote_samples" / "15Min_w3s"
QUOTE_YEARS = ([2017, 2018, 2019], list(range(2020, 2026)))       # the two S05b-method sample draws (P2-04 and S05b)
SCFG = s05.S05Config(close_long=1.0, close_short=-1.0)          # x10 exits (S05c)
OUT = ROOT / "notebooks" / "News_Filtered_Reversal" / ("02_synthetic" if SYNTHETIC else "02")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / "p4_02"
CACHE.mkdir(parents=True, exist_ok=True)
PANEL_B = work_dir() / "results" / "p3" / "panel_15Min_2017_2019.npz"     # built by P3 from the P2-04 bars
YEARS = {"A 2020-2025": list(range(2020, 2026)), "B 2017-2019": [2017, 2018, 2019]}
if SYNTHETIC:
    YEARS, N_PLACEBO = {"A 2025": [2025], "B 2024": [2024]}, 6
if SYNTHETIC:
    from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01
    ETF_OF = {s: SECTOR_ETF_01[k] for k, v in UNIVERSE_01.items() for s in v}
else:
    constituents = pd.read_csv(ROOT / "src" / "equity_trading" / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
STOCKS, ETFS = sorted(ETF_OF), sorted(set(ETF_OF.values()))
print(f"synthetic={SYNTHETIC} | exits x10 | placebos per window={N_PLACEBO}")

synthetic=False | exits x10 | placebos per window=100


## 1. Earnings events (cached by notebook 01)

In [2]:
if SYNTHETIC:
    EVENTS = None
else:
    EVENTS = p4.edgar_earnings(STOCKS, start="2016-01-01", end="2025-12-31", verbose=False)
    print(f"{len(EVENTS):,} earnings 8-Ks for {EVENTS['symbol'].nunique()} stocks")

19,921 earnings 8-Ks for 498 stocks


## 2. Per window: backtests, Q1, placebos, gates

In [3]:
from joblib import Parallel, delayed
RESULTS = {}
for wi, (wname, years) in enumerate(YEARS.items()):
    t0 = time.time()
    if SYNTHETIC:
        panel, _ = s04.synthetic_ohlc_panel(years, seed=3 + wi)
        H = np.full(panel.c.shape, 2e-4, np.float32)
    else:
        if wname.startswith("A"):
            panel = s04.cached_panel(STOCKS + ETFS + ["SPY"], SCFG.timeframe, years, ffill_bars=SCFG.ffill_bars)
        else:
            panel = p2.panel_cache(STOCKS + ETFS + ["SPY"], SCFG.timeframe, years, PANEL_B, ffill_bars=SCFG.ffill_bars)
        sess = s05b.sample_sessions(years, 4, seed=42)
        samples = pd.concat([pd.read_parquet(QUOTE_CACHE / f"{d}.parquet") for d in sess], ignore_index=True)
        H = s05b.cost_array(panel.index, panel.day_start, panel.symbols, s05b.cost_model(samples))
    have = set(panel.symbols)
    TRADED = [s for s in STOCKS if s in have and ETF_OF[s] in have]
    SC = np.array([panel.symbols.index(s) for s in TRADED]); EC = np.array([panel.symbols.index(ETF_OF[s]) for s in TRADED])
    tag = ("synthetic_" if SYNTHETIC else "") + f"{wname.split()[0]}_s05{s05.S05_VERSION}_p{s04.PANEL_VERSION}"
    est_path = CACHE / f"params_{tag}.npz"
    P = dict(np.load(est_path)) if est_path.exists() else s05.estimate_all(panel, SC, EC, SCFG, s05.first_decision_day(SCFG))
    if not est_path.exists():
        np.savez(est_path, **P)
    days = pd.DatetimeIndex(pd.to_datetime(list(panel.days)))
    ev = p4.synthetic_events(days, TRADED, seed=11 + wi) if SYNTHETIC else EVENTS
    EF = p4.event_flags(ev, days, TRADED, 1, 1).to_numpy()
    INSIDE = np.vstack([EF[1:], np.zeros((1, len(TRADED)), bool)])     # mask row e governs entries on session e + 1
    MASK = ~INSIDE
    print(f"{wname}: panel {panel.c.shape}, traded {len(TRADED)}, entries blocked on {INSIDE.mean():.2%} of stock-days ({time.time() - t0:.0f}s)", flush=True)
    runs, rows = {}, {}
    for arm, m in (("ALL", None), ("EX", MASK)):
        for cm in (1.0, 2.0, 0.0):
            runs[(arm, cm)] = s05.backtest(panel, P, SC, EC, replace(SCFG, cost_mult=cm), record=(cm == 1.0), half_spread=H, entry_mask=m)
    START = runs[("ALL", 1.0)]["first_day"]
    for (arm, cm), r in runs.items():
        rows[f"{arm} {cm:g}x"] = s04.perf_metrics(r["daily_equity"], SCFG.initial_equity, START)
    METRICS = pd.DataFrame(rows).T
    tr = runs[("ALL", 1.0)]["trades"].copy()
    lrow = p1.trade_label_rows(tr["entry_bar"].to_numpy(), panel.day_start)
    ok = lrow >= 0
    tr, lrow = tr[ok].copy(), lrow[ok]
    tr["inside"] = INSIDE[lrow, tr["i"].to_numpy()].astype(int); tr["entry_day"] = lrow
    BYW = tr.groupby("inside").agg(trades=("gross_ret", "size"), mean_gross_bps=("gross_ret", lambda x: 1e4 * x.mean()))
    Q1 = p1.cluster_bootstrap_diff(tr["gross_ret"].to_numpy(), tr["inside"].to_numpy(), tr["entry_day"].to_numpy(), 0, 1, 10_000, SCFG.seed, 0.05)
    pl_path = CACHE / f"placebo_{tag}_{N_PLACEBO}.parquet"
    if pl_path.exists():
        PL = pd.read_parquet(pl_path)
    else:
        t1 = time.time()
        seeds = [SCFG.seed * 1000 + wi * 10_000 + i for i in range(N_PLACEBO)]
        pool = np.ones(INSIDE.shape, bool)
        masks = (~p4.random_exclusion(INSIDE, pool, np.random.default_rng(s)) for s in seeds)
        PL = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=1, verbose=5)(
            delayed(p1.run_mask_backtest)(panel, P, SC, EC, SCFG, m, s, START, H) for s, m in zip(seeds, masks)))
        PL.to_parquet(pl_path); print(f"{wname}: {N_PLACEBO} placebos in {(time.time() - t1) / 60:.1f} min", flush=True)
    em = METRICS.loc["EX 1x"]
    RANKS = {k: s04.placebo_rank(em[k], PL[k].to_numpy()) for k in ("total_return", "sharpe")}
    e = runs[("EX", 1.0)]["daily_equity"]; e = e[e.index >= START]
    BOOT = {b: s04.block_bootstrap_mean(e.pct_change().dropna().to_numpy(), b, 10_000, SCFG.seed) for b in (10, 20, 40)}
    G = p4.gates_15m(Q1, RANKS, METRICS.loc["EX 1x"].to_dict(), METRICS.loc["EX 2x"].to_dict(), BOOT)
    w = wname.split()[0]
    for arm in ("ALL", "EX"):
        runs[(arm, 1.0)]["daily_equity"].to_csv(OUT / f"daily_equity_{w}_{arm}.csv", header=["equity"], float_format="%.2f")
    PL.to_csv(OUT / f"placebo_random_exclusion_{w}.csv", index=False, float_format="%.6g")
    RESULTS[wname] = {"metrics": METRICS.to_dict(orient="index"), "by_window": BYW.to_dict(orient="index"),
                      "Q1_outside_minus_inside": {"mean": Q1[0], "lo95": Q1[1], "hi95": Q1[2]}, "ranks": RANKS,
                      "bootstrap_EX": {str(k): v for k, v in BOOT.items()}, "gates": G, "blocked_share": float(INSIDE.mean())}
    print(f"--- {wname}"); display(METRICS.round(4)); display(BYW.round(3))
    print(f"outside - inside gross per trade: {1e4 * Q1[0]:.2f} bps [{1e4 * Q1[1]:.2f}, {1e4 * Q1[2]:.2f}]")
    display(pd.DataFrame(RANKS).T.round(4)); display(pd.Series(G))
    del panel, H, P, runs, PL

A 2020-2025: panel (39064, 513), traded 501, entries blocked on 4.91% of stock-days (37s)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   29.7s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  2.4min


A 2020-2025: 100 placebos in 3.7 min


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.7min finished


--- A 2020-2025


,total_return,sharpe,max_drawdown,mean_daily,days
ALL 1x,0.0987,0.3605,0.0782,0.0001,1486.0
ALL 2x,-0.0991,-0.2965,0.1918,-0.0001,1486.0
ALL 0x,0.3343,0.9937,0.0562,0.0002,1486.0
EX 1x,0.2141,0.6554,0.0827,0.0001,1486.0
EX 2x,-0.0045,0.0309,0.1154,0.0000,1486.0
EX 0x,0.4962,1.3078,0.0691,0.0003,1486.0


... (10 outputs left out)

,trades,mean_gross_bps
inside,,
0,5802,8.179
1,1973,-17.720


outside - inside gross per trade: 25.90 bps [1.21, 49.46]


,value,percentile,p_upper,placebo_median,placebo_p95
total_return,-0.0100,100.0,0.0099,-0.0574,-0.0452
sharpe,-0.1082,100.0,0.0099,-0.6733,-0.5206


Q1_outside_minus_inside     True
Q2_vs_random_exclusion      True
Q3_economic                False
pass                       False
dtype: bool

## 3. Verdict

In [4]:
VERDICT = "EDGE_CANDIDATE" if all(r["gates"]["pass"] for r in RESULTS.values()) else "NO_EDGE"
print("verdict:", VERDICT)
summary = {"study": "P4-02", "synthetic": SYNTHETIC, "years": YEARS, "config": SCFG.as_dict(), "p4_config": CFG.as_dict(),
           "n_placebo": N_PLACEBO, "results": RESULTS, "verdict": VERDICT,
           "prospective_verdict": "PENDING_2026_HOLDOUT" if VERDICT == "EDGE_CANDIDATE" else "NOT_APPLICABLE",
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "s05": s05.S05_VERSION, "panel": s04.PANEL_VERSION, "p4": p4.P4_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))

verdict: NO_EDGE


293